# VibNet-1D — Smoke Test de Portabilidade e Pipeline 1D no Domínio do Tempo (Google Colab)

Este notebook valida a portabilidade ponta a ponta do repositório **`VibNet-1D`** em uma instância limpa do **Google Colab** (ou servidor Linux) e executa os testes iniciais da fase **1D Time-Domain** (janelas físicas de 1 segundo, *detrend*, normalização *Z-Score*/*RMS* e *resampling* polifásico).

### Roteiro de Validação
1. **Setup do Ambiente e Instalação:** Clonagem (quando necessário), instalação editável (`pip install -e .`), instalação do fork `vibdata` e detecção de GPU.
2. **Conexão com Google Drive e `src.config`:** Redirecionamento dinâmico de `RAW_DATA_DIR`, `DATA_ROOT` e `RESULTS_DIR` via variáveis de ambiente sem uso de `sys.path.append`.
3. **Download e Janelamento de 1 Segundo (CWRU):** Extração de janelas de 1s baseadas em `fs`, aplicação de `Detrend`, normalizações (`Z-Score` e `RMS`) e `resampling`.
4. **Sanity Check Visual e Shapes PyTorch:** Comparação gráfica com `matplotlib` e verificação do formato `[batch, 1, seq_len]`.
5. **Forward Pass (Modelo 1D Time-Domain & TabVibNet):** Execução ponta a ponta passando pelo `load_vibration_data`, `build_1d_time_model` e pipeline tabular.

## 1. Setup do Ambiente e Instalação

Instala o pacote `vibnet1d` (`src`) em modo editável (`pip install -e .`) e o fork da biblioteca `vibdata`, além de verificar a disponibilidade de aceleração por GPU via PyTorch.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vfrocha/VibNet-1D.git"

# Se estiver no Google Colab fora do repositório, clona e entra na raiz do projeto
if IN_COLAB and not Path("pyproject.toml").exists():
    if not Path("/content/VibNet-1D").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/VibNet-1D"], check=True)
    os.chdir("/content/VibNet-1D")
elif Path("../pyproject.toml").exists() and not Path("pyproject.toml").exists():
    os.chdir("..")

print(f"Diretório de trabalho atual: {Path.cwd()}")

# Instala dependências, o fork da vibdata e o projeto em modo editável (pip install -e .)
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/vfrocha/vibdata.git
!pip install -q -e .

import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"PyTorch versão : {torch.__version__}")
print(f"GPU disponível : {CUDA_AVAILABLE} ({torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'CPU'})")

## 2. Conexão com Google Drive e Configuração de Caminhos (`src.config`)

Define as variáveis de ambiente `VIBNET_RAW_DIR`, `VIBNET_PROCESSED_DIR` e `VIBNET_RESULTS_DIR` **antes** de importar `src.config`. Isso comprova que o repositório resolve os diretórios de forma 100% portátil sem depender de caminhos fixos ou de `sys.path.append`.

In [ ]:
import importlib
import os
from pathlib import Path

# Altere para True no Google Colab caso deseje persistir dados/resultados no seu Google Drive
MOUNT_GOOGLE_DRIVE = False

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_WORKSPACE = Path("/content/drive/MyDrive/VibNet_Project")
else:
    BASE_WORKSPACE = Path("/content/vibnet_workspace") if IN_COLAB else (Path.cwd() / "data" / "smoke_test_workspace")

# Define variáveis de ambiente antes de carregar src.config
os.environ["VIBNET_RAW_DIR"] = str(BASE_WORKSPACE / "raw_data")
os.environ["VIBNET_PROCESSED_DIR"] = str(BASE_WORKSPACE / "processed")
os.environ["VIBNET_RESULTS_DIR"] = str(BASE_WORKSPACE / "results")

# Importação direta do pacote instalado (sem sys.path.append!)
import src.config as config
importlib.reload(config)
from src.config import DATA_ROOT, PROJECT_ROOT, RAW_DATA_DIR, RESULTS_DIR

for directory in (RAW_DATA_DIR, DATA_ROOT, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("=== Caminhos Resolvidos via src.config ===")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR : {RAW_DATA_DIR}")
print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")

assert RAW_DATA_DIR == (BASE_WORKSPACE / "raw_data").resolve()
assert DATA_ROOT == (BASE_WORKSPACE / "processed").resolve()
assert RESULTS_DIR == (BASE_WORKSPACE / "results").resolve()
print("\n[OK] Redirecionamento de caminhos via variáveis de ambiente validado com sucesso!")

## 3. Teste Rápido de Download e Janelamento de 1 Segundo (Base de Controle CWRU)

Carrega sinais da base de controle **CWRU** via `vibdata.raw.CWRU_raw(..., download=True)` (com *fallback* sintético caso executado em ambiente *offline*), extrai janelas de **exatamente 1 segundo físico** de acordo com a taxa de amostragem (`fs`), e aplica as transformações de `src.data.transforms_1d`:
1. **Detrend Linear (`detrend_1d`)**: Remoção de tendência linear / offset DC.
2. **Z-Score na Janela (`zscore_normalize_1d`)**: Média $0$ e desvio padrão $1$ por janela.
3. **Normalização por RMS (`rms_normalize_1d`)**: Energia RMS unitária ou escalada por `reference_rms`.
4. **Resampling Polifásico (`resample_1d`)**: Harmonização da taxa de amostragem para uma frequência alvo fixa (`target_fs = 6000 Hz`).

In [ ]:
import numpy as np
import pandas as pd

from src.data.transforms_1d import (
    TimeDomainTransform1D,
    detrend_1d,
    resample_1d,
    rms_normalize_1d,
    zscore_normalize_1d,
)

DOWNLOAD_CWRU = True  # Define se deve baixar a CWRU real via vibdata
TARGET_FS = 6000      # Frequência alvo fixa (Hz) para teste de resampling

raw_windows_1s = []
window_labels = []
window_conditions = []
orig_fs = 12000

if DOWNLOAD_CWRU:
    try:
        import vibdata.raw as raw_datasets
        from src.data.make_dataset import SimpleSplit, extract_signal, get_names
        print("Inicializando CWRU_raw via vibdata (download=True)...")
        cwru_ds = raw_datasets.CWRU_raw(str(RAW_DATA_DIR), download=True)
        
        # Extrai janelas de 1s das primeiras gravações de 12 kHz para um smoke test rápido
        for idx in range(min(len(cwru_ds), 12)):
            item = cwru_ds[idx]
            sig = extract_signal(item)
            if sig is None:
                continue
            meta = item["metainfo"].iloc[0] if isinstance(item["metainfo"], pd.DataFrame) else item["metainfo"]
            sr = int(meta.get("sample_rate", 12000))
            if sr != 12000:
                continue
            orig_fs = sr
            splitter = SimpleSplit(window_size=orig_fs, overlap=0)
            wins = splitter.transform({"signal": sig})["signal"]
            cond, lbl = get_names("CWRU_12k", meta)
            for w in wins[:5]:
                raw_windows_1s.append(np.asarray(w, dtype=np.float32))
                window_conditions.append(cond)
                window_labels.append(lbl)
        print(f"[OK] Extraídas {len(raw_windows_1s)} janelas reais de 1s ({orig_fs} pontos) da CWRU.")
    except Exception as exc:
        print(f"[AVISO] Download/leitura CWRU indisponível ({exc}). Gerando sinal de controle sintético CWRU 12k...")

# Fallback determinístico caso a máquina esteja sem rede ou o download seja desativado
if len(raw_windows_1s) < 8:
    rng = np.random.default_rng(42)
    orig_fs = 12000
    t = np.linspace(0.0, 1.0, orig_fs, endpoint=False, dtype=np.float32)
    raw_windows_1s, window_conditions, window_labels = [], [], []
    for cond in ["Load_0HP", "Load_1HP"]:
        for cls_idx, lbl in enumerate(["Class_Normal", "Class_Inner_Race", "Class_Outer_Race"]):
            for _ in range(4):
                drift = 0.25 + 0.15 * t  # Offset DC + tendência linear
                carrier = (0.3 + 0.4 * cls_idx) * np.sin(2 * np.pi * 60.0 * t)
                impulses = (0.5 * cls_idx) * np.sin(2 * np.pi * 162.0 * t) * (1.0 + 0.5 * np.cos(2 * np.pi * 30.0 * t))
                noise = rng.normal(0.0, 0.15, size=orig_fs).astype(np.float32)
                raw_windows_1s.append(drift + carrier + impulses + noise)
                window_conditions.append(cond)
                window_labels.append(lbl)
    print(f"[OK] Geradas {len(raw_windows_1s)} janelas de controle de 1s ({orig_fs} Hz).")

# Salva as janelas processadas (detrended) em DATA_ROOT/CWRU_12k para o teste do Dataloader na Célula 5
for i, (w, cond, lbl) in enumerate(zip(raw_windows_1s, window_conditions, window_labels)):
    out_dir = DATA_ROOT / "CWRU_12k" / cond / lbl
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"smoke_w{i:04d}.npy", detrend_1d(w))

# Seleciona uma janela de 1 segundo para testar todas as transformações 1D
sample_raw = raw_windows_1s[0]
sample_detrend = detrend_1d(sample_raw)
sample_zscore = zscore_normalize_1d(sample_detrend)
sample_rms = rms_normalize_1d(sample_detrend)
sample_resampled = resample_1d(sample_zscore, orig_fs=orig_fs, target_fs=TARGET_FS)

def _stats(x: np.ndarray) -> str:
    rms_val = float(np.sqrt(np.mean(np.square(x))))
    return f"len={len(x):5d} | mean={np.mean(x):+.4f} | std={np.std(x):.4f} | RMS={rms_val:.4f}"

print("\n=== Verificação Numérica na Janela de 1 Segundo ===")
print(f"1. Sinal Bruto (com offset) : {_stats(sample_raw)}")
print(f"2. Detrend (sem offset DC)  : {_stats(sample_detrend)}")
print(f"3. Normalização Z-Score     : {_stats(sample_zscore)}")
print(f"4. Normalização por RMS     : {_stats(sample_rms)}")
print(f"5. Resampled ({TARGET_FS} Hz)      : {_stats(sample_resampled)}")

assert len(sample_detrend) == orig_fs
assert len(sample_resampled) == TARGET_FS
assert np.isclose(np.mean(sample_zscore), 0.0, atol=1e-5)
assert np.isclose(np.std(sample_zscore), 1.0, atol=1e-4)
assert np.isclose(np.sqrt(np.mean(sample_rms ** 2)), 1.0, atol=1e-4)

## 4. Visualização Comparativa (Sanity Check Visual) e Shapes PyTorch

Plota a mesma janela de 1 segundo antes e depois das normalizações e do *resampling* (com zoom de `50 ms` para inspeção de fase e morfologia), e valida a conversão para tensor PyTorch no formato esperado pelos modelos convolucionais 1D: `[batch_size, 1, seq_len]`.

In [ ]:
import matplotlib.pyplot as plt

t_orig = np.linspace(0.0, 1.0, len(sample_detrend), endpoint=False)
t_res = np.linspace(0.0, 1.0, len(sample_resampled), endpoint=False)
zoom_mask_orig = t_orig <= 0.05  # Primeiros 50 ms para visualização nítida
zoom_mask_res = t_res <= 0.05

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=False)

axes[0, 0].plot(t_orig, sample_raw, alpha=0.6, label="Bruto (c/ offset)", color="tab:gray")
axes[0, 0].plot(t_orig, sample_detrend, alpha=0.85, label="Detrend (1s)", color="tab:blue")
axes[0, 0].set_title(f"Janela Completa de 1s ({orig_fs} Hz) — Bruto vs. Detrend")
axes[0, 0].set_xlabel("Tempo (s)")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].legend(loc="upper right")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(t_orig[zoom_mask_orig], sample_detrend[zoom_mask_orig], label="Detrend", color="tab:blue")
axes[0, 1].plot(t_orig[zoom_mask_orig], sample_zscore[zoom_mask_orig], label="Z-Score", color="tab:orange", alpha=0.8)
axes[0, 1].set_title("Zoom 50 ms — Detrend vs. Normalização Z-Score")
axes[0, 1].set_xlabel("Tempo (s)")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].legend(loc="upper right")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(t_orig[zoom_mask_orig], sample_rms[zoom_mask_orig], label="Divisão por RMS (RMS=1)", color="tab:green")
axes[1, 0].set_title("Zoom 50 ms — Normalização por Energia RMS")
axes[1, 0].set_xlabel("Tempo (s)")
axes[1, 0].set_ylabel("Amplitude / RMS")
axes[1, 0].legend(loc="upper right")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(t_orig[zoom_mask_orig], sample_zscore[zoom_mask_orig], label=f"Original ({orig_fs} Hz)", color="tab:blue", alpha=0.7)
axes[1, 1].plot(t_res[zoom_mask_res], sample_resampled[zoom_mask_res], "--", label=f"Resampled ({TARGET_FS} Hz)", color="tab:red")
axes[1, 1].set_title(f"Zoom 50 ms — Fidelidade do Resampling ({orig_fs} Hz -> {TARGET_FS} Hz)")
axes[1, 1].set_xlabel("Tempo (s)")
axes[1, 1].set_ylabel("Amplitude (Z-Score)")
axes[1, 1].legend(loc="upper right")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verificação dos shapes dos tensores PyTorch [batch, 1, seq_len]
pipeline_1d = TimeDomainTransform1D(
    apply_detrend=True,
    orig_fs=orig_fs,
    target_fs=TARGET_FS,
    normalization="window_zscore",
)

batch_np = np.stack([pipeline_1d(w) for w in raw_windows_1s[:8]], axis=0)
batch_tensor = torch.from_numpy(batch_np).unsqueeze(1).to(DEVICE)  # [batch, 1, seq_len]

print(f"Shape do array NumPy processado : {batch_np.shape}  -> (batch, seq_len)")
print(f"Shape do tensor PyTorch 1D      : {tuple(batch_tensor.shape)} -> (batch, channels=1, seq_len)")
assert batch_tensor.shape == (len( batch_np), 1, TARGET_FS)

## 5. Teste de Forward Pass (Modelo 1D Time-Domain e TabVibNet)

Executa um teste completo carregando janelas via `load_vibration_data` (usando `DATA_ROOT` de `src.config`) e realizando o *forward pass* tanto na arquitetura 1D no domínio do tempo (`build_1d_time_model`) quanto no extrator + classificador tabular (`TabVibNet` / `MLPClassifier`).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

from src.data.dataloader import load_vibration_data
from src.features.extractors import extract_freq_features, extract_time_features
from src.models.build_1d_time import build_1d_time_model
from src.models.build_mlp_contrastive import MLPClassifier

# 1. Carrega os dados usando o dataloader padrão (que importa DATA_ROOT de src.config)
test_cond = window_conditions[0]
X_train_raw, y_train, X_test_raw, y_test, label_encoder = load_vibration_data(
    dataset_name="CWRU_12k",
    test_condition=test_cond,
    task="diagnosis",
)

# Caso todas as janelas baixadas pertençam à mesma condição, usa um lote direto para o smoke test
if len(X_train_raw) == 0:
    X_train_raw = np.stack([detrend_1d(w) for w in raw_windows_1s], axis=0)
    y_train = np.arange(len(X_train_raw)) % 3
    num_classes = 3
else:
    num_classes = max(int(len(label_encoder.classes_)), 2)

# 2. Forward Pass — Modelo 1D Time-Domain (src.models.build_1d_time)
X_1d = np.stack([pipeline_1d(w) for w in X_train_raw[:8]], axis=0)
y_1d = torch.tensor(y_train[: len(X_1d)], dtype=torch.long)
ds_1d = TensorDataset(torch.from_numpy(X_1d).unsqueeze(1), y_1d)
loader_1d = DataLoader(ds_1d, batch_size=4, shuffle=False)

model_1d = build_1d_time_model(num_classes=num_classes, embedding_dim=128, device=DEVICE)
model_1d.eval()

xb, yb = next(iter(loader_1d))
xb = xb.to(DEVICE)
with torch.no_grad():
    latent_1d = model_1d.encode(xb)
    logits_1d = model_1d(xb)

print("\n=== [1] Forward Pass: Modelo 1D no Domínio do Tempo ===")
print(f"Input batch shape   : {tuple(xb.shape)} -> [batch, 1, seq_len]")
print(f"Latent vector shape : {tuple(latent_1d.shape)} -> [batch, embedding_dim]")
print(f"Output logits shape : {tuple(logits_1d.shape)} -> [batch, num_classes]")
assert logits_1d.shape == (xb.shape[0], num_classes)

# 3. Forward Pass — Pipeline Tabular (TabVibNet: Extratores + Multi-Head MLP)
feats_time = extract_time_features(X_train_raw[:8])
feats_freq = extract_freq_features(X_train_raw[:8], fs=orig_fs)
X_tab = np.hstack([feats_time, feats_freq]).astype(np.float32)
X_tab_tensor = torch.from_numpy(X_tab).to(DEVICE)

tab_model = MLPClassifier(
    input_dim=X_tab.shape[1],
    n_classes={"CWRU_12k": num_classes},
    encoder_profile="baseline",
).to(DEVICE)
tab_model.eval()

with torch.no_grad():
    logits_tab = tab_model(X_tab_tensor, "CWRU_12k")

print("\n=== [2] Forward Pass: Pipeline TabVibNet (Features + Multi-Head MLP) ===")
print(f"Tabular features shape : {tuple(X_tab_tensor.shape)} -> [batch, n_features]")
print(f"Tabular logits shape   : {tuple(logits_tab.shape)} -> [batch, num_classes]")
assert logits_tab.shape == (X_tab.shape[0], num_classes)

print("\n✅ SMOKE TEST CONCLUÍDO COM SUCESSO! O repositório VibNet-1D está 100% portátil e pronto para o Colab.")